In [1]:
import pandas as pd

ts_data = pd.read_parquet("../data/transformed/transformed_ts_2026_05.parquet")
ts_data

,pickup_hour,rides,pickup_location_id
0,2026-05-01 00:00:00,1,2
1,2026-05-01 01:00:00,0,2
2,2026-05-01 02:00:00,0,2
3,2026-05-01 03:00:00,0,2
4,2026-05-01 04:00:00,0,2
...,...,...,...
192691,2026-05-31 19:00:00,0,176
192692,2026-05-31 20:00:00,0,176
192693,2026-05-31 21:00:00,0,176
192694,2026-05-31 22:00:00,0,176


In [2]:
ts_data_one_location = ts_data.loc[ts_data.pickup_location_id == 43, :].reset_index(drop=True)
ts_data_one_location.head(25)

,pickup_hour,rides,pickup_location_id
0,2026-05-01 00:00:00,7,43
1,2026-05-01 01:00:00,4,43
2,2026-05-01 02:00:00,1,43
3,2026-05-01 03:00:00,1,43
4,2026-05-01 04:00:00,2,43
5,2026-05-01 05:00:00,7,43
6,2026-05-01 06:00:00,20,43
7,2026-05-01 07:00:00,22,43
8,2026-05-01 08:00:00,56,43
9,2026-05-01 09:00:00,62,43


In [3]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int,
    step_size: int
    ) -> list:

        stop_position = len(data) - 1
        
        # Start the first sub-sequence at index position 0
        subseq_first_idx = 0
        subseq_mid_idx = n_features
        subseq_last_idx = n_features + 1
        indices = []
        
        while subseq_last_idx <= stop_position:
            indices.append((subseq_first_idx, subseq_mid_idx, subseq_last_idx))
            
            subseq_first_idx += step_size
            subseq_mid_idx += step_size
            subseq_last_idx += step_size

        return indices

In [4]:
n_features = 24
step_size = 1

indices = get_cutoff_indices(ts_data_one_location, n_features, step_size)
indices[:5]

[(0, 24, 25), (1, 25, 26), (2, 26, 27), (3, 27, 28), (4, 28, 29)]

In [5]:
import numpy as np
print(np.__version__)

n_examples = len(indices)
x = np.ndarray(shape=(n_examples, n_features), dtype=np.float32)
y = np.ndarray(shape=(n_examples), dtype=np.float32)
pickup_hours = []

for i, idx in enumerate(indices):
    x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
    y[i] = ts_data_one_location.iloc[idx[1]]['rides']  # <-- fixed: scalar access
    pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

2.4.3


In [6]:
print(f'{x.shape=}')
print(f'{x=}')
print(f'{pickup_hours[:5]=}')

x.shape=(719, 24)
x=array([[  7.,   4.,   1., ...,  64.,  60.,  38.],
       [  4.,   1.,   1., ...,  60.,  38.,  28.],
       [  1.,   1.,   2., ...,  38.,  28.,  12.],
       ...,
       [135., 109.,  63., ..., 107., 105.,  75.],
       [109.,  63.,  49., ..., 105.,  75.,  69.],
       [ 63.,  49.,  30., ...,  75.,  69.,  52.]],
      shape=(719, 24), dtype=float32)
pickup_hours[:5]=[Timestamp('2026-05-02 00:00:00'), Timestamp('2026-05-02 01:00:00'), Timestamp('2026-05-02 02:00:00'), Timestamp('2026-05-02 03:00:00'), Timestamp('2026-05-02 04:00:00')]


In [7]:
features_one_location = pd.DataFrame(
    x,
    columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(n_features))]
)
features_one_location

,rides_previous_24_hour,rides_previous_23_hour,rides_previous_22_hour,rides_previous_21_hour,rides_previous_20_hour,rides_previous_19_hour,rides_previous_18_hour,rides_previous_17_hour,rides_previous_16_hour,rides_previous_15_hour,...,rides_previous_10_hour,rides_previous_9_hour,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour
0,7.0,4.0,1.0,1.0,2.0,7.0,20.0,22.0,56.0,62.0,...,132.0,180.0,184.0,152.0,137.0,128.0,105.0,64.0,60.0,38.0
1,4.0,1.0,1.0,2.0,7.0,20.0,22.0,56.0,62.0,72.0,...,180.0,184.0,152.0,137.0,128.0,105.0,64.0,60.0,38.0,28.0
2,1.0,1.0,2.0,7.0,20.0,22.0,56.0,62.0,72.0,133.0,...,184.0,152.0,137.0,128.0,105.0,64.0,60.0,38.0,28.0,12.0
3,1.0,2.0,7.0,20.0,22.0,56.0,62.0,72.0,133.0,145.0,...,152.0,137.0,128.0,105.0,64.0,60.0,38.0,28.0,12.0,2.0
4,2.0,7.0,20.0,22.0,56.0,62.0,72.0,133.0,145.0,114.0,...,137.0,128.0,105.0,64.0,60.0,38.0,28.0,12.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,155.0,119.0,135.0,109.0,63.0,49.0,30.0,11.0,4.0,3.0,...,22.0,32.0,38.0,55.0,57.0,40.0,54.0,79.0,75.0,107.0
715,119.0,135.0,109.0,63.0,49.0,30.0,11.0,4.0,3.0,2.0,...,32.0,38.0,55.0,57.0,40.0,54.0,79.0,75.0,107.0,105.0
716,135.0,109.0,63.0,49.0,30.0,11.0,4.0,3.0,2.0,1.0,...,38.0,55.0,57.0,40.0,54.0,79.0,75.0,107.0,105.0,75.0
717,109.0,63.0,49.0,30.0,11.0,4.0,3.0,2.0,1.0,2.0,...,55.0,57.0,40.0,54.0,79.0,75.0,107.0,105.0,75.0,69.0


In [8]:
targets_one_location = pd.DataFrame(y, columns=[f'target_rides_next_hour'])
targets_one_location

,target_rides_next_hour
0,28.0
1,12.0
2,2.0
3,1.0
4,0.0
...,...
714,105.0
715,75.0
716,69.0
717,52.0


In [9]:
from tqdm import tqdm

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'pickup_hour', 'rides', 'pickup_location_id'}

    location_ids = ts_data['pickup_location_id'].unique()
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for location_id in tqdm(location_ids):
        
        # keep only ts data for this `location_id`
        ts_data_one_location = ts_data.loc[
            ts_data.pickup_location_id == location_id, 
            ['pickup_hour', 'rides']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        pickup_hours = []
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
            y[i] = ts_data_one_location.iloc[idx[1]]['rides']
            # y[i] = ts_data_one_location.iloc[idx[1]:idx[2]]['rides'].values
            pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

        # numpy -> pandas
        features_one_location = pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        features_one_location['pickup_hour'] = pickup_hours
        features_one_location['pickup_location_id'] = location_id

        # numpy -> pandas
        targets_one_location = pd.DataFrame(y, columns=[f'target_rides_next_hour'])

        # concatenate results
        features = pd.concat([features, features_one_location])
        targets = pd.concat([targets, targets_one_location])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_rides_next_hour']

In [10]:
features, targets = transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len=24*7*1, # one week of history
    step_size=24,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

100%|██████████| 259/259 [00:00<00:00, 687.79it/s]

features.shape=(6216, 170)
targets.shape=(6216,)


In [11]:
features.head()

,rides_previous_168_hour,rides_previous_167_hour,rides_previous_166_hour,rides_previous_165_hour,rides_previous_164_hour,rides_previous_163_hour,rides_previous_162_hour,rides_previous_161_hour,rides_previous_160_hour,rides_previous_159_hour,...,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,pickup_location_id
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-05-08,2
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-05-09,2
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-05-10,2
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-05-11,2
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-05-12,2


In [12]:
targets.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: target_rides_next_hour, dtype: float32